In [24]:
from config.data import *

base_path = f"{TICKER}_{START_DATE}_to_{END_DATE}_{TARGET_FEATURE}_ret"
data_path = f"cache/processed_data/{base_path}.pkl"

meta_path = f"{base_path}_seq-{LAG_DAYS}-step_1_{MODEL_NAME}_layers{LAYERS}_dropout{DROPOUT}_epochs{EPOCHS}_bs{BATCH_SIZE}"
model_path = f"cache/trained_models/{meta_path}.keras"

plot_path = f"results/{meta_path}_predictions_chart.png"
report_path = f"results/{meta_path}.csv"


# Preprocessed data

# Scaler

In [25]:
import pickle
data = pickle.load(open(f"{data_path}", 'rb'))
data  # Will show formatted output

{'X_train': array([[[ 0.41712   ,  0.38002808,  0.42909011,  0.40539346,
          -0.32506988],
         [ 0.5107625 ,  0.49488919,  0.49216897,  0.4572539 ,
          -0.16939223],
         [ 0.44646979,  0.45798617,  0.47414649,  0.49603272,
           0.11629314],
         ...,
         [ 1.04373692,  1.02814471,  1.04091756,  1.06883899,
          -0.49572555],
         [ 1.01438639,  0.97140569,  0.98874644,  0.94969955,
          -0.24855806],
         [ 1.16440149,  1.11994171,  1.12059727,  1.08238801,
          -0.23802209]],
 
        [[-1.47028012, -1.48342652, -1.48659997, -1.46329437,
          -1.07848367],
         [-1.46129717, -1.46697225, -1.47562612, -1.47680678,
          -0.78944467],
         [-1.44108661, -1.45763376, -1.44819271, -1.4632945 ,
          -0.73296418],
         ...,
         [-1.09884742, -1.11031976, -1.09406842, -1.111972  ,
          -0.33674001],
         [-1.03731687, -1.06006866, -1.04720307, -1.06152631,
           0.56352242],
         [-1

In [26]:

# Inspect a target scaler
scalers = data['scalers']
tf_key = data['target_feature'].lower()
target_scaler = scalers[f"{TICKER}_{tf_key}"]

print(f"{tf_key} mean:", target_scaler.mean_)
print(f"{tf_key} scale:", target_scaler.scale_)
print(f"{tf_key} var:", target_scaler.var_)

# Inspect all scalers
for key, scaler in data['scalers'].items():
    print(f"{key}: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}")

close_return mean: [0.00120633]
close_return scale: [0.01189326]
close_return var: [0.00014145]
CBA.AX_close: mean=124.1612, std=20.8457
CBA.AX_high: mean=124.8556, std=21.0533
CBA.AX_low: mean=123.1475, std=20.4765
CBA.AX_open: mean=124.0412, std=20.7864
CBA.AX_volume: mean=2051690.6436, std=961942.8351
CBA.AX_close_return: mean=0.0012, std=0.0119


# Model

In [ ]:
import tensorflow as tf #type: ignore

model = tf.keras.models.load_model(model_path)
model.summary()

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, None, 50)       │        11,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, None, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, None, 50)       │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, None, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 154,955 (605.30 KB)

 Trainable params: 51,651 (201.76 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 103,304 (403.54 KB)

# Prediction

In [ ]:
import pandas as pd #type: ignore

from model.tf_models import TFModel

data = pd.read_pickle(data_path)
print("type:", type(data))
print(list(data.keys()))
x_test = data['X_test']
y_test = data['y_test']
x_train = data['X_train']
# Init model instance
input_size = x_train.shape[2] # no_features
tf_model = TFModel(
    input_size=input_size,
    model_name=MODEL_NAME,
    layers=LAYERS,
    dropout_rate=float(DROPOUT),
)
scalers = data['scalers']
tf_key = data['target_feature'].lower()
target_scaler = scalers[f"{TICKER}_{tf_key}"]

keras_model = tf.keras.models.load_model(model_path)
tf_model.model = keras_model  # Replace the built model with loaded one

model = tf_model
predictions, metrics = model.predict_and_evaluate(x_test, y_test)

# Inverse transform using separate scaler for target feature
if target_scaler is not None:
    actual_prices = target_scaler.inverse_transform(
        y_test.reshape(-1, 1)
    ).reshape(-1)
    predicted_prices = target_scaler.inverse_transform(
        predictions.reshape(-1, 1)
    ).reshape(-1)
    
else:
    # No scaling was applied, use values as-is
    actual_prices = y_test.reshape(-1)
    predicted_prices = predictions.reshape(-1)
    print("No inverse transform applied - values used as-is")
    
print(f"Actual prices: {actual_prices[:10]}")
print(f"Predicted prices: {predicted_prices[:10]}")

import numpy as np #type: ignore
mae_price = float(np.mean(np.abs(actual_prices - predicted_prices)))
rmse_price = float(np.sqrt(np.mean((actual_prices - predicted_prices) ** 2)))
print("loss (scaled MSE):", float(metrics['loss']))
print("mae (price):", mae_price)
print("rmse (price):", rmse_price)


type: <class 'dict'>
['X_train', 'X_test', 'y_train', 'y_test', 'test_df', 'training_features', 'target_feature', 'scalers', 'seed_price']
Actual prices: [ 0.00806669  0.01204121  0.04100794 -0.02550093  0.00578407 -0.01141748
  0.00079785  0.02196831  0.00389398  0.01430673]
Predicted prices: [ 0.00891544  0.0061972   0.00291577  0.0019211   0.00121262 -0.00023804
 -0.00171252 -0.00406761 -0.00425431 -0.004702  ]
loss (scaled MSE): 1.418707013130188
mae (price): 0.011022837594473483
rmse (price): 0.014166000525472506
